# Using the 96 head

Some liquid handling robots have a 96 head, which can be used to pipette 96 samples at once. On the Hamilton STAR, the 96 head is an optional module. After {meth}`~pylabrobot.hamilton.liquid_handlers.star.star.STAR.setup`, it is available as `star.head96` (or `None` if not installed).

This notebook shows how to use the {class}`~pylabrobot.capabilities.liquid_handling.head96.Head96` capability to pick up tips, aspirate, dispense, and work with 384-well plate quadrants.

## Setup

Import {class}`~pylabrobot.hamilton.liquid_handlers.star.star.STAR` and create a {class}`~pylabrobot.resources.hamilton.decks.STARLetDeck`. After `setup()`, `star.head96` is a {class}`~pylabrobot.capabilities.liquid_handling.head96.Head96` instance if the hardware is installed.

In [ ]:
from pylabrobot.hamilton.liquid_handlers.star import STAR
from pylabrobot.resources.hamilton import STARLetDeck

deck = STARLetDeck()
star = STAR(deck=deck)
await star.setup()

## Creating the deck layout

Assign a tip carrier with a 96-tip rack and a plate carrier with a 96-well plate.

In [ ]:
from pylabrobot.resources import (
    TIP_CAR_480_A00,
    PLT_CAR_L5AC_A00,
    TIP_50ul,
    Cor_96_wellplate_360ul_Fb,
)

tip_carrier = TIP_CAR_480_A00(name="tip_carrier")
tip_carrier[1] = tip_rack = TIP_50ul(name="tip_rack")
deck.assign_child_resource(tip_carrier, rails=1)

plt_carrier = PLT_CAR_L5AC_A00(name="plt_carrier")
plt_carrier[0] = plate = Cor_96_wellplate_360ul_Fb(name="plate")
deck.assign_child_resource(plt_carrier, rails=7)

## Liquid handling with the 96 head

Liquid handling with the 96 head is very similar to what you would do with individual channels. The methods live on `star.head96` and take {class}`~pylabrobot.resources.tip_rack.TipRack`s and {class}`~pylabrobot.resources.plate.Plate`s as arguments, as opposed to individual `TipSpot`s and `Well`s used with `star.pip`.

In [ ]:
await star.head96.pick_up_tips(tip_rack)

For aspirations and dispenses, a single volume is passed because all 96 channels move in unison.

```{note}
Only single-volume aspirations and dispenses are supported because all robots that are currently implemented only support single-volume operations. When we add support for robots that can do variable-volume, this will be updated.
```

In [ ]:
await star.head96.aspirate(plate, volume=50)

In [ ]:
await star.head96.dispense(plate, volume=50)

In [ ]:
await star.head96.return_tips()

## Quadrants

96 heads can also be used to pipette quadrants of a 384-well plate. A 384-well plate is laid out as four interleaved 96-well quadrants. Use {meth}`~pylabrobot.resources.plate.Plate.get_quadrant` to select the wells for a given quadrant (1 through 4).

![quadrants](img/96head/quadrants.png)

In [ ]:
from pylabrobot.resources import BioRad_384_DWP_50uL_Vb

plt_carrier[1] = plate384 = BioRad_384_DWP_50uL_Vb(name="plate384")

In [ ]:
await star.head96.pick_up_tips(tip_rack)

In [ ]:
await star.head96.aspirate(plate384.get_quadrant(1), volume=10)

In [ ]:
await star.head96.dispense(plate384.get_quadrant(2), volume=10)

In [ ]:
await star.head96.aspirate(plate384.get_quadrant(3), volume=10)

In [ ]:
await star.head96.dispense(plate384.get_quadrant(4), volume=10)

In [ ]:
await star.head96.return_tips()

## Backend parameters

For STAR-specific tuning, pass `backend_params` to any 96-head operation. The available parameter classes are:

- {class}`~pylabrobot.hamilton.liquid_handlers.star.head96_backend.STARHead96Backend.PickUpTips96Params`
- {class}`~pylabrobot.hamilton.liquid_handlers.star.head96_backend.STARHead96Backend.DropTips96Params`
- {class}`~pylabrobot.hamilton.liquid_handlers.star.head96_backend.STARHead96Backend.Aspirate96Params`
- {class}`~pylabrobot.hamilton.liquid_handlers.star.head96_backend.STARHead96Backend.Dispense96Params`

For example, to customize the tip pickup method:

In [ ]:
from pylabrobot.hamilton.liquid_handlers.star.head96_backend import STARHead96Backend

await star.head96.pick_up_tips(
    tip_rack,
    backend_params=STARHead96Backend.PickUpTips96Params(
        tip_pickup_method="from_rack",
    ),
)

await star.head96.aspirate(plate, volume=50)
await star.head96.dispense(plate, volume=50)
await star.head96.return_tips()

## Teardown

In [ ]:
await star.stop()